In [0]:
from delta.tables import DeltaTable
import pyspark.sql.functions as F

In [0]:
BASE_PATH = "/Volumes/workspace/project_data_football_raw/mercado_raw"

In [0]:
# Lista os arquivos no diretório BASE_PATH
arquivos = dbutils.fs.ls(BASE_PATH)

# Ordena os arquivos pelo nome em ordem decrescente
arquivos = sorted(arquivos, key=lambda x: x.name, reverse=True)

# Verifica se há arquivos disponíveis
if not arquivos:
    raise Exception("Nenhum arquivo encontrado na landing de mercado")

# Seleciona o caminho do arquivo mais recente
ultimo_arquivo = arquivos[0].path

# Exibe o caminho do arquivo selecionado
print(f"Arquivo: {ultimo_arquivo}")

df_raw = spark.read.json(ultimo_arquivo)

In [0]:
# Explodimos um atleta para pegar o ID da rodada vinculado a ele
primeiro_atleta = df_raw.select(F.explode("atletas").alias("a")).select("a.rodada_id").first()

if primeiro_atleta:
    rodada_atual = primeiro_atleta["rodada_id"]
else:
    # Backup: se o JSON estiver vindo de uma estrutura diferente, tenta pegar do nome do arquivo
    match = re.search(r'(\d+)', ultimo_arquivo)
    rodada_atual = int(match.group(1)) if match else 0

print(f"Rodada referência detectada: {rodada_atual}")

status_exemplo = df_raw.select("status").first()
print(f"Exemplo de dados em 'status': {status_exemplo}")

In [0]:
# CLUBES

# Transformamos o STRUCT de clubes em MAP para permitir o explode
df_clubes_map = df_raw.select(
    F.explode(
        F.from_json(F.to_json(F.col("clubes")), "map<string,string>")
    ).alias("clube_id", "json")
)

# Pegamos o esquema do primeiro clube disponível na estrutura para usar como molde
primeiro_id_clube = df_raw.schema["clubes"].dataType.names[0]
clube_schema = df_raw.schema["clubes"].dataType[primeiro_id_clube].dataType

# Monta o DataFrame final de clubes, convertendo o JSON para o esquema correto e adicionando colunas de referência
df_clubes = df_clubes_map.select(
    F.col("clube_id").cast("int"),
    F.from_json(F.col("json"), clube_schema).alias("d")
).select("clube_id", "d.*") \
 .withColumn("rodada_referencia", F.lit(rodada_atual).cast("int")) \
 .withColumn("dt_ingestao", F.current_timestamp())
 
# Nome da tabela Delta bronze de clubes
tabela_clubes = "project_data_football_bronze.clubes"

# Verifica se a tabela já existe para decidir entre merge ou criação
if spark.catalog.tableExists(tabela_clubes):
    dt_clubes = DeltaTable.forName(spark, tabela_clubes)
    # Realiza merge: atualiza registros existentes ou insere novos
    dt_clubes.alias("target").merge(
        df_clubes.alias("source"),
        "target.clube_id = source.clube_id AND target.rodada_referencia = source.rodada_referencia"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print("Clubes mesclados com sucesso!")
else:
    # Cria a tabela Delta bronze de clubes
    df_clubes.write\
        .format("delta")\
        .saveAsTable("project_data_football_bronze.clubes")

In [0]:
# POSICOES

# 1. Converte STRUCT para MAP para habilitar o explode
df_posicoes = df_raw.select(
    F.explode(
        F.from_json(F.to_json(F.col("posicoes")), "map<string,string>")
    ).alias("posicao_id", "posicao_json_str")
)

# 2. Captura o esquema dinâmico das posições
primeira_pos_id = df_raw.schema["posicoes"].dataType.names[0]
posicao_schema = df_raw.schema["posicoes"].dataType[primeira_pos_id].dataType

df_posicoes = df_posicoes.select(
    F.col("posicao_id").cast("int"),
    F.from_json(F.col("posicao_json_str"), posicao_schema).alias("d")
).select("posicao_id", "d.*") \
 .withColumn("rodada_referencia", F.lit(rodada_atual).cast("int")) \
 .withColumn("dt_ingestao", F.current_timestamp())

# Nome da tabela Delta bronze de posições
tabela_posicoes = "project_data_football_bronze.posicoes"

# Verifica se a tabela já existe para decidir entre merge ou criação
if spark.catalog.tableExists(tabela_posicoes):
    dt_posicoes = DeltaTable.forName(spark, tabela_posicoes)
    # Realiza merge: atualiza registros existentes ou insere novos
    dt_posicoes.alias("target").merge(
        df_posicoes.alias("source"),
        "target.posicao_id = source.posicao_id AND target.rodada_referencia = source.rodada_referencia"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print("Posições mescladas com sucesso!")
else:
    # Cria a tabela Delta bronze de posições
    df_posicoes.write \
        .format("delta") \
        .saveAsTable("project_data_football_bronze.posicoes")

In [0]:
# ATLETAS

# Explode o campo 'atletas' para obter cada atleta como uma linha e seleciona todos os campos do atleta
df_atletas = df_raw.select(F.explode("atletas").alias("a")).select("a.*") \
    .withColumn("rodada_referencia", F.lit(rodada_atual).cast("int")) \
    .withColumn("dt_ingestao", F.current_timestamp())

tabela_atletas = "project_data_football_bronze.atletas"

# Verifica se a tabela já existe para decidir entre merge ou criação
if spark.catalog.tableExists(tabela_atletas):
    dt_atleta = DeltaTable.forName(spark, tabela_atletas)
    
    # Merge para evitar duplicados
    dt_atleta = DeltaTable.forName(spark, tabela_atletas)
    dt_atleta.alias("target").merge(
        df_atletas.alias("source"),
        "target.atleta_id = source.atleta_id AND target.rodada_referencia = source.rodada_referencia"
    )\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()
    print("Atletas mesclados com sucesso!")
else:
    # Cria a tabela Delta bronze de atletas
    df_atletas.write \
        .format("delta") \
        .saveAsTable("project_data_football_bronze.atletas")